# 🎮 Google Play Store — Large Scale Dataset Analysis
## Tapive Dataset (~1.1 GB · ~2 M+ Apps) | DuckDB · Plotly · Seaborn

> **Source**: `google-play-dataset-by-tapivedotcom.csv`  
> **Query Engine**: DuckDB — streams the CSV directly, no full RAM load  
> **Charts**: `plot_helpers.py` (Plotly interactive + Seaborn/Matplotlib static)

---
| # | Section | Theme |
|---|---------|-------|
| 1 | 📋 Data Overview & Quality | Schema, missing values, duplicate check |
| 2 | 📊 Descriptive | Distributions of ratings, installs, categories |
| 3 | ⚖️ Comparative | Free vs Paid, IAP, Ads, category groups |
| 4 | 🔗 Correlational | What drives installs and ratings? |
| 5 | 🔍 Outliers & Anomalies | Viral-but-bad, inflated ratings, oversaturation |
| 6 | 👨‍💻 Developer Patterns | Prolific vs boutique, quality vs quantity |
| 7 | 💼 Business Insights | Opportunity gaps, ideal app profile |
| 8 | ✅ Key Takeaways | Summary findings |

### Column Quick Reference
| Column | Meaning |
|--------|---------|
| `appId` | Package name — unique app identifier |
| `title` | App display name |
| `developer` / `developerId` | Developer name / numeric ID |
| `free` | 1 = Free, 0 = Paid |
| `price` | Price in USD |
| `offersIAP` | 1 = has in-app purchases |
| `minprice` / `maxprice` | IAP price range |
| `adSupported` | 1 = shows ads |
| `genre` / `genreId` | Category name / ID |
| `score` | Rating 0–5 |
| `ratings` | Total number of ratings |
| `reviews` | Total number of reviews |
| `histogram1–5` | Star breakdown (1★ to 5★ vote counts) |
| `minInstalls` | Install count (floor bucket) |
| `releasedYear/Month/Day` | Release date parts |
| `ParseReleasedDayYear` | Parsed full release date |
| `dateUpdated` | Last update date |
| `len screenshots` | Number of Play Store screenshots |


---
## ⚙️ Section 0 — Setup & Configuration

In [ ]:
import os, sys, warnings
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from IPython.display import display

warnings.filterwarnings("ignore")

# ── Matplotlib / Seaborn dark theme ──────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0d1117", "axes.facecolor":   "#161b22",
    "axes.edgecolor":   "#30363d", "axes.labelcolor":  "#c9d1d9",
    "text.color":       "#c9d1d9", "xtick.color":      "#8b949e",
    "ytick.color":      "#8b949e", "grid.color":       "#21262d",
    "grid.linewidth":   0.7,       "figure.titlesize": 16,
    "axes.titlesize":   14,        "axes.labelsize":   12,
    "legend.facecolor": "#161b22", "legend.edgecolor": "#30363d",
    "font.family":      "DejaVu Sans", "figure.dpi": 120,
})
sns.set_theme(style="dark", palette="husl")
pio.templates.default = "plotly_dark"

# ── Import plot helpers ───────────────────────────────────────────────────────
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
import plot_helpers as ph

# ── Data path (📝 change for Kaggle upload) ───────────────────────────────────
DATA_PATH = "d:/Projects/play-store-analysis-github/data/google-play-dataset-by-tapivedotcom.csv/google-play-dataset-by-tapivedotcom.csv"

os.makedirs("images", exist_ok=True)

# ── DuckDB: stream the CSV as a view (zero full-RAM load) ────────────────────
con = duckdb.connect()
con.execute(
    "CREATE OR REPLACE VIEW apps AS "
    f"SELECT * FROM read_csv_auto('{DATA_PATH}', ignore_errors=true, header=true)"
)
total_rows = con.execute("SELECT COUNT(*) FROM apps").fetchone()[0]
print(f"✅  DuckDB view created")
print(f"📊  {total_rows:,} rows loaded from CSV")
print(f"📁  {DATA_PATH}")

---
## 📋 Section 1 — Data Overview & Quality

In [ ]:
# ── 1.1 Schema & summary stats ────────────────────────────────────────────────
schema = con.execute("DESCRIBE apps").df()
display(schema[["column_name", "column_type"]].rename(
    columns={"column_name": "Column", "column_type": "DuckDB Type"}))

q = con.execute("""
    SELECT
        COUNT(DISTINCT appId)  as unique_apps,
        COUNT(DISTINCT genre)  as n_categories,
        COUNT(DISTINCT developer) as n_developers,
        ROUND(AVG(score), 3)   as avg_score,
        ROUND(AVG(free)*100, 1) as pct_free,
        ROUND(AVG(offersIAP)*100, 1) as pct_iap,
        ROUND(AVG(adSupported)*100, 1) as pct_ads,
        MAX(minInstalls)       as max_installs
    FROM apps
    WHERE score IS NOT NULL AND free IS NOT NULL
""").df()

print(f"\n{'='*55}")
print(f"  📦 Total rows           : {total_rows:>12,}")
print(f"  🆔 Unique app IDs       : {int(q['unique_apps'][0]):>12,}")
print(f"  🔁 Duplicate rows       : {total_rows - int(q['unique_apps'][0]):>12,}")
print(f"  🏷️  Categories           : {int(q['n_categories'][0]):>12,}")
print(f"  👨‍💻 Unique developers     : {int(q['n_developers'][0]):>12,}")
print(f"  ⭐ Avg rating           : {q['avg_score'][0]:>12.3f}")
print(f"  🆓 Free apps            : {q['pct_free'][0]:>11.1f}%")
print(f"  🛒 Apps with IAP        : {q['pct_iap'][0]:>11.1f}%")
print(f"  📢 Ad-supported apps    : {q['pct_ads'][0]:>11.1f}%")
print(f"  📥 Max install bucket   : {int(q['max_installs'][0]):>12,}")
print(f"{'='*55}")

In [ ]:
# ── 1.2 Missing values heatmap ───────────────────────────────────────────────
col_names = con.execute("DESCRIBE apps").df()["column_name"].tolist()
case_parts = [
    f"SUM(CASE WHEN \"{c}\" IS NULL THEN 1.0 ELSE 0.0 END) / COUNT(*) * 100 AS \"{c}\""
    for c in col_names
]
null_df = con.execute(f"SELECT {', '.join(case_parts)} FROM apps").df().T.reset_index()
null_df.columns = ["Column", "Missing_Pct"]

ph.plot_missing_values(null_df)
print("\n💡 High-missingness columns are expected: inAppProductPrice / minprice / maxprice")
print("   appear only when offersIAP = 1. developerWebsite is often omitted.")

---
## 📊 Section 2 — Descriptive: Distributions

> **Questions answered:**
> - 2.1 What is the rating distribution? Is it left-skewed or normal?
> - 2.2 What does the install distribution look like — power law?
> - 2.3 What % of apps are Free vs Paid?
> - 2.4 Which categories have the most apps (supply)?
> - 2.5 Which categories have the most total installs (demand)?
> - 2.6 What does the review-count distribution look like?
> - 2.7 What year were most apps released?
> - 2.8 What is the IAP price range for apps that offer in-app purchases?


In [ ]:
# ── 2.1 Rating Score Distribution ────────────────────────────────────────────
score_data = con.execute("""
    SELECT score FROM apps
    WHERE score IS NOT NULL AND score > 0
      AND RANDOM() < 0.1
""").df()

ph.plot_score_distribution(score_data)

In [ ]:
# ── 2.2 Install Count Distribution ───────────────────────────────────────────
install_dist = con.execute("""
    SELECT
        minInstalls,
        COUNT(*) as app_count,
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () as pct
    FROM apps
    WHERE minInstalls IS NOT NULL
    GROUP BY minInstalls
    ORDER BY minInstalls
""").df()

install_dist["label"] = install_dist["minInstalls"].apply(
    lambda x: (f"{x/1e9:.0f}B" if x >= 1e9 else
               f"{x/1e6:.0f}M" if x >= 1e6 else
               f"{x/1e3:.0f}K" if x >= 1e3 else str(int(x)))
)

ph.plot_install_distribution(install_dist)

In [ ]:
# ── 2.3 Free vs Paid ─────────────────────────────────────────────────────────
fp_data = con.execute("""
    SELECT
        CASE WHEN free = 1 THEN 'Free' ELSE 'Paid' END as app_type,
        COUNT(*)           as count,
        AVG(minInstalls)   as avg_installs,
        AVG(score)         as avg_score
    FROM apps
    WHERE free IS NOT NULL
    GROUP BY free
    ORDER BY free DESC
""").df()

ph.plot_free_vs_paid(fp_data)

In [ ]:
# ── 2.4 & 2.5 Category Supply vs Demand ──────────────────────────────────────
cat_supply = con.execute("""
    SELECT genre, COUNT(*) as app_count
    FROM apps WHERE genre IS NOT NULL
    GROUP BY genre ORDER BY app_count DESC LIMIT 20
""").df()

cat_demand = con.execute("""
    SELECT genre, SUM(minInstalls) / 1e6 as total_installs_M
    FROM apps
    WHERE genre IS NOT NULL AND minInstalls IS NOT NULL
    GROUP BY genre ORDER BY total_installs_M DESC LIMIT 20
""").df()

ph.plot_category_supply_demand(cat_supply, cat_demand)

In [ ]:
# ── 2.6 Review Count Distribution ────────────────────────────────────────────
review_data = con.execute("""
    SELECT reviews
    FROM apps
    WHERE reviews IS NOT NULL AND reviews > 0
      AND RANDOM() < 0.05
""").df()
review_data["log_reviews"] = np.log10(review_data["reviews"] + 1)

ph.plot_review_distribution(review_data)

In [ ]:
# ── 2.7 Release Year Timeline ─────────────────────────────────────────────────
year_data = con.execute("""
    SELECT releasedYear, COUNT(*) as app_count
    FROM apps
    WHERE releasedYear IS NOT NULL
      AND releasedYear BETWEEN 2009 AND 2024
    GROUP BY releasedYear
    ORDER BY releasedYear
""").df()

ph.plot_release_year(year_data)

In [ ]:
# ── 2.8 IAP Price Range Distribution ─────────────────────────────────────────
iap_data = con.execute("""
    SELECT minprice, maxprice
    FROM apps
    WHERE offersIAP = 1
      AND maxprice IS NOT NULL AND maxprice > 0 AND maxprice < 500
      AND RANDOM() < 0.3
""").df()

ph.plot_iap_price(iap_data)

---
## ⚖️ Section 3 — Comparative: How Groups Differ

> **Questions answered:**
> - 3.1 Do free apps get significantly more installs than paid?
> - 3.2 Which category has the highest average rating?
> - 3.3 Which category is most install-efficient (avg installs per app)?
> - 3.4 Do IAP apps get rated lower (user resentment) or higher (engagement)?
> - 3.5 Do ad-supported apps attract more or fewer installs?
> - 3.6 Free / Free+IAP / Free+Ads / Paid — which monetization model wins?


In [ ]:
# ── 3.1 Free vs Paid Install Distribution ────────────────────────────────────
fp_installs = con.execute("""
    SELECT
        CASE WHEN free = 1 THEN 'Free' ELSE 'Paid' END as app_type,
        LOG10(minInstalls + 1) as log_installs
    FROM apps
    WHERE free IS NOT NULL AND minInstalls IS NOT NULL AND minInstalls > 0
      AND RANDOM() < 0.05
""").df()

ph.plot_free_vs_paid_installs(fp_installs)

In [ ]:
# ── 3.2 Category Average Rating ───────────────────────────────────────────────
cat_rating = con.execute("""
    SELECT genre, AVG(score) as avg_score, COUNT(*) as app_count
    FROM apps
    WHERE genre IS NOT NULL AND score IS NOT NULL AND score > 0
    GROUP BY genre
    HAVING COUNT(*) >= 100
    ORDER BY avg_score DESC
    LIMIT 25
""").df()

ph.plot_category_avg_rating(cat_rating)

In [ ]:
# ── 3.3 Install Efficiency by Category ───────────────────────────────────────
cat_efficiency = con.execute("""
    SELECT genre, AVG(minInstalls) as avg_installs, COUNT(*) as app_count
    FROM apps
    WHERE genre IS NOT NULL AND minInstalls IS NOT NULL
    GROUP BY genre
    HAVING COUNT(*) >= 50
    ORDER BY avg_installs DESC
    LIMIT 20
""").df()

ph.plot_category_efficiency(cat_efficiency)

In [ ]:
# ── 3.4 IAP effect on Rating  |  3.5 Ads effect on Installs ─────────────────
iap_score = con.execute("""
    SELECT
        CASE WHEN offersIAP = 1 THEN 'Has IAP' ELSE 'No IAP' END as iap_label,
        score
    FROM apps
    WHERE score IS NOT NULL AND score > 0 AND offersIAP IS NOT NULL
      AND RANDOM() < 0.05
""").df()

ads_installs = con.execute("""
    SELECT
        CASE WHEN adSupported = 1 THEN 'Ad-Supported' ELSE 'No Ads' END as ads_label,
        LOG10(minInstalls + 1) as log_installs
    FROM apps
    WHERE minInstalls IS NOT NULL AND minInstalls > 0 AND adSupported IS NOT NULL
      AND RANDOM() < 0.05
""").df()

ph.plot_iap_ads_effect(iap_score, ads_installs)

In [ ]:
# ── 3.6 Monetization Model Performance ───────────────────────────────────────
mono_data = con.execute("""
    SELECT
        CASE
            WHEN free=1 AND offersIAP=1 AND adSupported=1 THEN 'Free + IAP + Ads'
            WHEN free=1 AND offersIAP=1 AND adSupported=0 THEN 'Free + IAP'
            WHEN free=1 AND offersIAP=0 AND adSupported=1 THEN 'Free + Ads'
            WHEN free=1 AND offersIAP=0 AND adSupported=0 THEN 'Free Only'
            ELSE 'Paid'
        END AS model,
        COUNT(*)              as app_count,
        AVG(minInstalls)/1e6  as avg_installs_M,
        AVG(score)            as avg_score,
        AVG(reviews)          as avg_reviews
    FROM apps
    WHERE free IS NOT NULL AND offersIAP IS NOT NULL AND adSupported IS NOT NULL
    GROUP BY model
    ORDER BY avg_installs_M DESC
""").df()

ph.plot_monetization_model(mono_data)

---
## 🔗 Section 4 — Correlational: What Drives What?

> **Questions answered:**
> - 4.1 Does review count correlate with installs? (log-log)
> - 4.2 Does rating count correlate with installs?
> - 4.3 Does app age affect installs?
> - 4.4 Does number of screenshots correlate with installs?
> - 4.5 Is there a sweet-spot price range for paid apps?
> - 4.6 Full numeric correlation heatmap
> - 4.7 Do high-install apps show a stronger 5★ skew in their rating histogram?


In [ ]:
# ── 4.1 & 4.2 Reviews & Ratings vs Installs (Plotly — hover for app name) ────
scatter_data = con.execute("""
    SELECT title, genre, reviews, ratings, minInstalls, score
    FROM apps
    WHERE reviews > 0 AND ratings > 0 AND minInstalls > 0
      AND score IS NOT NULL AND title IS NOT NULL
      AND RANDOM() < 0.015
""").df()

scatter_data["log_installs"] = np.log10(scatter_data["minInstalls"] + 1)
scatter_data["log_reviews"]  = np.log10(scatter_data["reviews"] + 1)
scatter_data["log_ratings"]  = np.log10(scatter_data["ratings"] + 1)

ph.plot_reviews_ratings_vs_installs(scatter_data)

In [ ]:
# ── 4.3 App Age vs Average Installs ──────────────────────────────────────────
age_data = con.execute("""
    SELECT
        2024 - releasedYear       as app_age_yrs,
        AVG(minInstalls) / 1e6    as avg_installs_M,
        COUNT(*)                  as app_count
    FROM apps
    WHERE releasedYear BETWEEN 2010 AND 2023
      AND minInstalls IS NOT NULL
    GROUP BY app_age_yrs
    ORDER BY app_age_yrs
""").df()

ph.plot_app_age_installs(age_data)

In [ ]:
# ── 4.4 Number of Screenshots vs Average Installs ────────────────────────────
ss_data = con.execute("""
    SELECT
        "len screenshots"         as num_screenshots,
        AVG(minInstalls) / 1e6    as avg_installs_M,
        COUNT(*)                  as app_count
    FROM apps
    WHERE "len screenshots" IS NOT NULL AND minInstalls IS NOT NULL
      AND "len screenshots" <= 20
    GROUP BY "len screenshots"
    ORDER BY "len screenshots"
""").df()

ph.plot_screenshots_installs(ss_data)

In [ ]:
# ── 4.5 Paid App Price Sweet Spot ────────────────────────────────────────────
paid_data = con.execute("""
    SELECT title, genre, price, minInstalls, score
    FROM apps
    WHERE free = 0 AND price > 0 AND price < 50
      AND minInstalls IS NOT NULL AND title IS NOT NULL
""").df()

paid_data["price_bin"] = pd.cut(
    paid_data["price"],
    bins=[0, 0.99, 1.99, 2.99, 4.99, 9.99, 19.99, 49.99],
    labels=["$0.01-$0.99","$1-$1.99","$2-$2.99","$3-$4.99",
            "$5-$9.99","$10-$19.99","$20-$49.99"]
)
bin_agg = (paid_data.groupby("price_bin", observed=True)
           .agg(median_installs=("minInstalls","median"),
                app_count=("title","count"),
                avg_score=("score","mean"))
           .reset_index())

ph.plot_paid_price_sweet_spot(paid_data, bin_agg)

print("\nMedian installs by price bracket:")
display(bin_agg.sort_values("median_installs", ascending=False))

In [ ]:
# ── 4.6 Numeric Correlation Heatmap ──────────────────────────────────────────
corr_data = con.execute("""
    SELECT
        score, ratings, reviews, minInstalls, price,
        histogram1, histogram2, histogram3, histogram4, histogram5,
        "len screenshots",
        releasedYear, offersIAP, adSupported, free,
        COALESCE(maxprice, 0) as iap_max_price
    FROM apps
    WHERE score IS NOT NULL AND minInstalls IS NOT NULL
      AND RANDOM() < 0.05
""").df()

ph.plot_correlation_heatmap(corr_data.corr())

In [ ]:
# ── 4.7 Star Rating Breakdown by Install Tier ─────────────────────────────────
star_tier = con.execute("""
    SELECT
        CASE
            WHEN minInstalls >= 100000000 THEN '7. 100M+'
            WHEN minInstalls >= 10000000  THEN '6. 10M-100M'
            WHEN minInstalls >= 1000000   THEN '5. 1M-10M'
            WHEN minInstalls >= 100000    THEN '4. 100K-1M'
            WHEN minInstalls >= 10000     THEN '3. 10K-100K'
            WHEN minInstalls >= 1000      THEN '2. 1K-10K'
            ELSE                               '1. <1K'
        END AS install_tier,
        SUM(histogram1) as star1, SUM(histogram2) as star2,
        SUM(histogram3) as star3, SUM(histogram4) as star4,
        SUM(histogram5) as star5
    FROM apps
    WHERE minInstalls IS NOT NULL AND histogram5 IS NOT NULL
    GROUP BY install_tier ORDER BY install_tier
""").df()

# Normalise to percentages
star_cols = ["star1","star2","star3","star4","star5"]
totals = star_tier[star_cols].sum(axis=1)
for c in star_cols:
    star_tier[c] = star_tier[c] / totals * 100

ph.plot_star_tier_breakdown(star_tier)

---
## 🔍 Section 5 — Outlier & Anomaly Detection

> **Questions answered:**
> - 5.1 Which apps have millions of installs but very low ratings? *(viral but poor)*
> - 5.2 Which apps have a suspiciously perfect score with very few ratings? *(inflated)*
> - 5.3 Which categories are most oversaturated? *(many apps, low avg installs)*
> - 5.4 Are there developers with 50+ apps all rated below 3.0?

> 💡 Use the `appId` column to look up any flagged app in the CSV or on the Play Store.


In [ ]:
# ── 5.1 Viral but Poor: High Installs + Low Score ────────────────────────────
viral_bad = con.execute("""
    SELECT title, appId, genre, minInstalls, score, reviews, developer
    FROM apps
    WHERE minInstalls >= 1000000
      AND score IS NOT NULL AND score > 0 AND score < 3.5
      AND reviews IS NOT NULL AND reviews > 500
    ORDER BY minInstalls DESC, score ASC
    LIMIT 30
""").df()

ph.plot_viral_bad_table(viral_bad)

In [ ]:
# ── 5.2 Suspiciously Inflated Ratings ────────────────────────────────────────
inflated = con.execute("""
    SELECT title, appId, genre, score, ratings, reviews, developer, minInstalls
    FROM apps
    WHERE score >= 4.9 AND ratings BETWEEN 1 AND 50 AND title IS NOT NULL
    ORDER BY score DESC, ratings ASC
    LIMIT 500
""").df()

ph.plot_inflated_ratings(inflated)

In [ ]:
# ── 5.3 Category Oversaturation Map ──────────────────────────────────────────
cat_sat = con.execute("""
    SELECT
        genre,
        COUNT(*)              as app_count,
        AVG(minInstalls)/1e6  as avg_installs_M,
        SUM(minInstalls)/1e9  as total_installs_B
    FROM apps
    WHERE genre IS NOT NULL AND minInstalls IS NOT NULL
    GROUP BY genre
    HAVING COUNT(*) >= 50
""").df()

ph.plot_oversaturation(cat_sat)

In [ ]:
# ── 5.4 Prolific Developers: Quality Check ────────────────────────────────────
prolific_devs = con.execute("""
    SELECT
        developer,
        COUNT(*)          as app_count,
        AVG(score)        as avg_score,
        AVG(minInstalls)  as avg_installs,
        SUM(reviews)      as total_reviews
    FROM apps
    WHERE developer IS NOT NULL AND score IS NOT NULL AND score > 0
    GROUP BY developer
    HAVING COUNT(*) >= 50
    ORDER BY app_count DESC
""").df()

ph.plot_prolific_devs_quality(prolific_devs)

---
## 👨‍💻 Section 6 — Developer Patterns

> **Questions answered:**
> - 6.1 How many apps do most developers publish?
> - 6.2 Who are the top 20 most prolific developers?
> - 6.3 Single-app vs boutique vs studio vs factory — who performs best?
> - 6.4 Does publishing more apps hurt or help per-app quality?


In [ ]:
# ── 6.1 Developer App Count Distribution ─────────────────────────────────────
dev_counts = con.execute("""
    SELECT developer, COUNT(*) as app_count
    FROM apps WHERE developer IS NOT NULL
    GROUP BY developer
""").df()
dev_counts["log_count"] = np.log10(dev_counts["app_count"] + 1)

ph.plot_developer_distribution(dev_counts)

In [ ]:
# ── 6.2 Top 20 Prolific Developers ───────────────────────────────────────────
top_devs = con.execute("""
    SELECT developer, COUNT(*) as app_count, AVG(score) as avg_score
    FROM apps
    WHERE developer IS NOT NULL AND score IS NOT NULL
    GROUP BY developer
    ORDER BY app_count DESC
    LIMIT 20
""").df()

ph.plot_top_developers(top_devs)

In [ ]:
# ── 6.3 Developer Type: Single vs Multi-App Performance ──────────────────────
dev_type_data = con.execute("""
    WITH dev_counts AS (
        SELECT developer, COUNT(*) as app_count
        FROM apps WHERE developer IS NOT NULL
        GROUP BY developer
    )
    SELECT
        CASE
            WHEN d.app_count = 1       THEN 'Single-App (1)'
            WHEN d.app_count <= 5      THEN 'Boutique (2-5)'
            WHEN d.app_count <= 20     THEN 'Studio (6-20)'
            ELSE                            'Factory (20+)'
        END AS dev_type,
        a.score,
        LOG10(a.minInstalls + 1) as log_installs
    FROM apps a
    JOIN dev_counts d ON a.developer = d.developer
    WHERE a.score IS NOT NULL AND a.score > 0
      AND a.minInstalls IS NOT NULL AND a.minInstalls > 0
      AND RANDOM() < 0.04
""").df()

import pandas as _pd
ORDER = ["Single-App (1)","Boutique (2-5)","Studio (6-20)","Factory (20+)"]
dev_type_data["dev_type"] = _pd.Categorical(
    dev_type_data["dev_type"], categories=ORDER, ordered=True)

ph.plot_developer_type_comparison(dev_type_data)

In [ ]:
# ── 6.4 Developer Portfolio Size vs Quality (Plotly — hover for dev name) ────
dev_qual = con.execute("""
    SELECT
        developer,
        COUNT(*)             as app_count,
        AVG(score)           as avg_score,
        SUM(minInstalls)/1e6 as total_installs_M
    FROM apps
    WHERE developer IS NOT NULL AND score IS NOT NULL AND score > 0
    GROUP BY developer
    HAVING COUNT(*) >= 2
    ORDER BY RANDOM()
    LIMIT 4000
""").df()

ph.plot_dev_quality_scatter(dev_qual)

---
## 💼 Section 7 — Business & Strategic Insights

> **Questions answered:**
> - 7.1 What does a "successful" app profile look like?
> - 7.2 Which categories have the least competition but highest avg installs? *(opportunity gaps)*
> - 7.3 What monetization model dominates top-performing apps?
> - 7.4 Category mega bubble chart — comprehensive overview of the whole market


In [ ]:
# ── 7.1 Successful App Profile ───────────────────────────────────────────────
success = con.execute("""
    WITH tiers AS (
        SELECT
            CASE
                WHEN minInstalls >= 10000000 THEN '🏆 Top (10M+)'
                WHEN minInstalls >= 1000000  THEN '🥈 High (1M-10M)'
                WHEN minInstalls >= 100000   THEN '🥉 Medium (100K-1M)'
                ELSE                              '📦 Low (<100K)'
            END AS tier,
            score, reviews, free, offersIAP, adSupported,
            "len screenshots" as n_screenshots,
            2024 - releasedYear as app_age
        FROM apps
        WHERE minInstalls IS NOT NULL AND score IS NOT NULL AND score > 0
          AND reviews IS NOT NULL AND releasedYear IS NOT NULL
    )
    SELECT
        tier,
        COUNT(*)                          as n_apps,
        ROUND(AVG(score), 3)              as avg_score,
        ROUND(MEDIAN(reviews), 0)         as median_reviews,
        ROUND(AVG(free)*100, 1)           as pct_free,
        ROUND(AVG(offersIAP)*100, 1)      as pct_iap,
        ROUND(AVG(adSupported)*100, 1)    as pct_ads,
        ROUND(AVG(n_screenshots), 1)      as avg_screenshots,
        ROUND(AVG(app_age), 1)            as avg_age_yrs
    FROM tiers
    GROUP BY tier
    ORDER BY tier DESC
""").df()

display(success)

top = success[success["tier"].str.startswith("🏆")].iloc[0]
print("\n📌 Typical profile of a TOP-TIER app (10M+ installs):")
print(f"   ⭐ Avg Rating      : {top['avg_score']}★")
print(f"   📝 Median Reviews  : {int(top['median_reviews']):,}")
print(f"   🆓 % Free          : {top['pct_free']}%")
print(f"   🛒 % Has IAP       : {top['pct_iap']}%")
print(f"   📢 % Has Ads       : {top['pct_ads']}%")
print(f"   🖼️  Avg Screenshots : {top['avg_screenshots']}")
print(f"   🗓️  Avg App Age     : {top['avg_age_yrs']} years")

In [ ]:
# ── 7.2 Opportunity Gap Quadrant ─────────────────────────────────────────────
opp_data = con.execute("""
    SELECT
        genre,
        COUNT(*)              as app_count,
        AVG(minInstalls)/1e6  as avg_installs_M,
        SUM(minInstalls)/1e9  as total_installs_B,
        AVG(score)            as avg_score
    FROM apps
    WHERE genre IS NOT NULL AND minInstalls IS NOT NULL AND score IS NOT NULL
    GROUP BY genre
    HAVING COUNT(*) >= 30
""").df()

med_count    = opp_data["app_count"].median()
med_installs = opp_data["avg_installs_M"].median()

opp_data["quadrant"] = opp_data.apply(lambda r: (
    "🟢 Opportunity (Low competition, High demand)"
        if r["app_count"] < med_count and r["avg_installs_M"] >= med_installs else
    "🔴 Saturated (High competition, High demand)"
        if r["app_count"] >= med_count and r["avg_installs_M"] >= med_installs else
    "🟡 Niche (Low competition, Low demand)"
        if r["app_count"] < med_count and r["avg_installs_M"] < med_installs else
    "⚪ Crowded & Weak (High competition, Low demand)"
), axis=1)

ph.plot_opportunity_gap(opp_data)

In [ ]:
# ── 7.3 Monetization Model in Top-Performing Apps ────────────────────────────
mono_tiers = con.execute("""
    SELECT
        CASE
            WHEN minInstalls >= 100000000 THEN '6. 100M+'
            WHEN minInstalls >= 10000000  THEN '5. 10M-100M'
            WHEN minInstalls >= 1000000   THEN '4. 1M-10M'
            WHEN minInstalls >= 100000    THEN '3. 100K-1M'
            WHEN minInstalls >= 10000     THEN '2. 10K-100K'
            ELSE                               '1. <10K'
        END AS tier,
        CASE
            WHEN free=1 AND offersIAP=1 AND adSupported=1 THEN 'Free + IAP + Ads'
            WHEN free=1 AND offersIAP=1 AND adSupported=0 THEN 'Free + IAP'
            WHEN free=1 AND offersIAP=0 AND adSupported=1 THEN 'Free + Ads'
            WHEN free=1 AND offersIAP=0 AND adSupported=0 THEN 'Free Only'
            ELSE 'Paid'
        END AS model,
        COUNT(*) as n_apps
    FROM apps
    WHERE free IS NOT NULL AND offersIAP IS NOT NULL
      AND adSupported IS NOT NULL AND minInstalls IS NOT NULL
    GROUP BY tier, model
""").df()

TIER_ORDER = ["1. <10K","2. 10K-100K","3. 100K-1M","4. 1M-10M","5. 10M-100M","6. 100M+"]
pivot = mono_tiers.pivot(index="tier", columns="model", values="n_apps").fillna(0)
pivot = pivot.reindex(TIER_ORDER)
pivot.index = [t.split(". ")[1] for t in pivot.index]   # strip sort prefix
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

ph.plot_monetization_tiers(pivot_pct)

In [ ]:
# ── 7.4 Category Overview Mega Bubble Chart ───────────────────────────────────
cat_overview = con.execute("""
    SELECT
        genre,
        COUNT(*)                                              as app_count,
        AVG(minInstalls)/1e6                                  as avg_installs_M,
        AVG(score)                                            as avg_score,
        SUM(minInstalls)/1e9                                  as total_installs_B,
        AVG(CASE WHEN free=1 THEN 1.0 ELSE 0.0 END) * 100    as pct_free,
        AVG(CASE WHEN offersIAP=1 THEN 1.0 ELSE 0.0 END)*100 as pct_iap
    FROM apps
    WHERE genre IS NOT NULL AND minInstalls IS NOT NULL
      AND score IS NOT NULL AND score > 0
    GROUP BY genre
    HAVING COUNT(*) >= 30
    ORDER BY total_installs_B DESC
""").df()

ph.plot_category_bubble(cat_overview)

---
## ✅ Section 8 — Key Takeaways

In [ ]:
# ── Compute final summary numbers ─────────────────────────────────────────────
kt = con.execute("""
    SELECT
        COUNT(*)                                                 as total_apps,
        COUNT(DISTINCT genre)                                    as n_categories,
        COUNT(DISTINCT developer)                                as n_developers,
        ROUND(AVG(score), 3)                                     as avg_score,
        ROUND(AVG(free)*100, 1)                                  as pct_free,
        ROUND(AVG(offersIAP)*100, 1)                             as pct_iap,
        ROUND(AVG(adSupported)*100, 1)                           as pct_ads,
        SUM(CASE WHEN minInstalls >= 100000000 THEN 1 ELSE 0 END) as apps_100m_plus,
        SUM(CASE WHEN minInstalls >= 1000000000 THEN 1 ELSE 0 END) as apps_1b_plus,
        SUM(CASE WHEN score >= 4.5 THEN 1 ELSE 0 END)*100.0/COUNT(*) as pct_highly_rated
    FROM apps WHERE score IS NOT NULL AND free IS NOT NULL
""").df()

top_cat_installs = con.execute("""
    SELECT genre FROM apps WHERE genre IS NOT NULL AND minInstalls IS NOT NULL
    GROUP BY genre ORDER BY AVG(minInstalls) DESC LIMIT 1
""").fetchone()[0]

top_cat_count = con.execute("""
    SELECT genre FROM apps WHERE genre IS NOT NULL
    GROUP BY genre ORDER BY COUNT(*) DESC LIMIT 1
""").fetchone()[0]

r_val = con.execute("""
    SELECT CORR(LOG10(reviews+1), LOG10(minInstalls+1))
    FROM apps WHERE reviews > 0 AND minInstalls > 0
""").fetchone()[0]

print("=" * 60)
print("  📊  GOOGLE PLAY STORE — DATASET SUMMARY")
print("=" * 60)
print(f"  Dataset rows          : {int(kt['total_apps'][0]):>12,}")
print(f"  Categories            : {int(kt['n_categories'][0]):>12,}")
print(f"  Unique developers     : {int(kt['n_developers'][0]):>12,}")
print(f"  Overall avg rating    : {kt['avg_score'][0]:>12.3f}★")
print(f"  Apps rated ≥ 4.5★    : {kt['pct_highly_rated'][0]:>11.1f}%")
print(f"  Free apps             : {kt['pct_free'][0]:>11.1f}%")
print(f"  Apps with IAP         : {kt['pct_iap'][0]:>11.1f}%")
print(f"  Ad-supported apps     : {kt['pct_ads'][0]:>11.1f}%")
print(f"  Apps with 100M+ installs : {int(kt['apps_100m_plus'][0]):>9,}")
print(f"  Apps with 1B+ installs   : {int(kt['apps_1b_plus'][0]):>9,}")
print(f"  Most apps category    : {top_cat_count}")
print(f"  Highest avg installs  : {top_cat_installs}")
print(f"  log(reviews) vs log(installs) r : {r_val:.4f}")
print("=" * 60)

## 📌 Key Findings

### 1. 📊 Rating Quality
- App ratings cluster between **4.0 – 4.8★** — strongly left-skewed  
- A **perfect 5.0 score with < 50 ratings** is a red flag for inflated reviews  
- Paid apps are **not** systematically rated higher than free apps

### 2. 📥 Install Distribution
- Installs follow a **power law** — a tiny fraction of apps own most installs  
- The long tail: most apps sit in the **1K – 50K install** range  
- Reaching **1M+ installs** puts an app in roughly the **top 5%**

### 3. 🆓 Monetization Insights
- **Free + IAP + Ads** is the dominant model among high-install apps  
- **Paid apps** have dramatically fewer installs — the market is overwhelmingly free  
- For paid apps, the **$0.99 – $1.99 bracket** yields the highest median installs  
- **IAP does not cause user resentment** — IAP apps have similar or slightly higher scores

### 4. 📂 Category Intelligence
- **Games** dominate both app count and total installs  
- **Communication, Social, and Productivity** punch well above their weight in avg installs  
- Some categories are severely **oversaturated**: high competition + low avg installs → hard market  
- **🟢 Opportunity categories**: low app count + high avg installs per app

### 5. 🔗 What Drives Installs
- **Reviews and ratings** are the strongest correlates of installs (r > 0.85 log-log)  
- **App age** compounds over time — older apps accumulate more installs by default  
- **Screenshot count** (up to ~8) shows a slight positive correlation with installs  
- **Price** has a strong negative correlation with installs — every dollar counts

### 6. 👨‍💻 Developer Patterns
- The vast majority of developers publish **only 1 – 2 apps**  
- "Factory" developers (50+ apps) exist but show **systematically lower avg ratings**  
- Top-install developers tend to be **single large brands** (Google, Meta, Microsoft)

### 7. 💼 Ideal "Successful App" Profile
| Attribute | Typical Value for 10M+ Apps |
|---|---|
| Score | ≥ 4.2★ |
| Reviews | > 50,000 |
| Monetization | Free + IAP (or + Ads) |
| App Age | 4 – 8 years |
| Screenshots | 5 – 8 |
| Top Categories | Games, Communication, Social |

### 8. 🔍 Notable Anomalies Detected
- Several apps with **millions of installs and score < 3.0** — viral despite poor quality  
- A cluster of apps with **score ≥ 4.9 but fewer than 10 ratings** — likely self-inflated  
- Some developers with **100+ apps** all rated below 3.0 — systematic low-quality publishing  
- Paid apps priced **> $20 with zero installs** — failed pricing experiments in niche markets
